# Social Media & Mental Health — Data Analysis

**Step 4 → 5 of the workflow**

| Step | Action |
|------|--------|
| 4 | Load data from SQLite into Pandas DataFrame |
| 5 | Answer research questions with statistical tests |

---

### Research Questions
- **RQ1**: Does screen time affect anxiety (GAD-7 score)?
- **RQ2**: Does late-night usage affect sleep duration?
- **RQ3**: Do some platforms have higher depression (PHQ-9) scores?
- **RQ4**: Does social comparison trigger affect mental health scores?
- **RQ5**: Which user archetypes are at highest mental health risk?

### Statistical Methods
T-Test · ANOVA · Chi-Square · Quartiles · Data Segmentation

## Setup

In [ ]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

DB_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'mental_health.db')
print('Database path:', DB_PATH)

## Step 4 — Load from SQLite into DataFrame

In [ ]:
conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    u.user_id,
    u.age,
    u.gender,
    a.archetype_name        AS user_archetype,
    p.platform_name         AS primary_platform,
    ub.daily_screen_time_hours,
    ct.content_type_name    AS dominant_content_type,
    ub.activity_type,
    ub.late_night_usage,
    ub.social_comparison_trigger,
    ub.sleep_duration_hours,
    mh.gad7_score,
    g.severity_label        AS gad7_severity,
    mh.phq9_score,
    ph.severity_label       AS phq9_severity
FROM users u
JOIN archetypes             a   ON u.archetype_id      = a.archetype_id
JOIN platforms              p   ON u.platform_id       = p.platform_id
JOIN usage_behavior         ub  ON u.user_id           = ub.user_id
JOIN content_types          ct  ON ub.content_type_id  = ct.content_type_id
JOIN mental_health_scores   mh  ON u.user_id           = mh.user_id
JOIN gad7_severity_ranges   g   ON mh.gad7_score  BETWEEN g.min_score  AND g.max_score
JOIN phq9_severity_ranges   ph  ON mh.phq9_score BETWEEN ph.min_score AND ph.max_score
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Cast boolean-like columns
df['late_night_usage']          = df['late_night_usage'].astype(bool)
df['social_comparison_trigger'] = df['social_comparison_trigger'].astype(bool)

# Ordered categories for severity
gad7_order  = ['Minimal', 'Mild', 'Moderate', 'Severe']
phq9_order  = ['None-Minimal', 'Mild', 'Moderate', 'Moderately Severe', 'Severe']
df['gad7_severity'] = pd.Categorical(df['gad7_severity'], categories=gad7_order,  ordered=True)
df['phq9_severity'] = pd.Categorical(df['phq9_severity'], categories=phq9_order, ordered=True)

df.info()

In [ ]:
df.describe()

## Step 5 — Analysis

---
### RQ1 — Does screen time affect anxiety (GAD-7)?
*Method: Quartile segmentation + ANOVA*

In [ ]:
# Quartile segmentation of daily screen time
df['screen_time_quartile'] = pd.qcut(
    df['daily_screen_time_hours'], q=4,
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
)

quartile_stats = df.groupby('screen_time_quartile', observed=True)['gad7_score'].agg(
    mean='mean', std='std', count='count'
).round(2)
print('GAD-7 by screen-time quartile:')
print(quartile_stats)

# One-way ANOVA
groups = [g['gad7_score'].values for _, g in df.groupby('screen_time_quartile', observed=True)]
f_stat, p_val = stats.f_oneway(*groups)
print(f'\nANOVA — F={f_stat:.3f}, p={p_val:.4f}')
print('Significant (p<0.05):', p_val < 0.05)

# Pearson correlation
r, p = stats.pearsonr(df['daily_screen_time_hours'], df['gad7_score'])
print(f'Pearson r={r:.3f}, p={p:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='screen_time_quartile', y='gad7_score', ax=axes[0])
axes[0].set_title('GAD-7 Score by Screen Time Quartile')
axes[0].set_xlabel('Screen Time Quartile')
axes[0].set_ylabel('GAD-7 Score')

sns.regplot(data=df, x='daily_screen_time_hours', y='gad7_score',
            scatter_kws={'alpha': 0.1}, ax=axes[1])
axes[1].set_title(f'Screen Time vs GAD-7 (r={r:.2f})')
axes[1].set_xlabel('Daily Screen Time (hours)')
axes[1].set_ylabel('GAD-7 Score')
plt.tight_layout()
plt.show()

---
### RQ2 — Does late-night usage affect sleep duration?
*Method: Independent-samples T-Test*

In [ ]:
late    = df[df['late_night_usage']]['sleep_duration_hours']
no_late = df[~df['late_night_usage']]['sleep_duration_hours']

print(f'Late-night users   — n={len(late):,}, mean sleep={late.mean():.2f} h, sd={late.std():.2f}')
print(f'No late-night      — n={len(no_late):,}, mean sleep={no_late.mean():.2f} h, sd={no_late.std():.2f}')

t_stat, p_val = stats.ttest_ind(late, no_late)
print(f'\nT-Test — t={t_stat:.3f}, p={p_val:.4f}')
print('Significant (p<0.05):', p_val < 0.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='late_night_usage', y='sleep_duration_hours', ax=axes[0])
axes[0].set_title('Sleep Duration vs Late-Night Usage')
axes[0].set_xticklabels(['No Late-Night', 'Late-Night'])
axes[0].set_ylabel('Sleep Duration (hours)')

sns.histplot(data=df, x='sleep_duration_hours', hue='late_night_usage',
             kde=True, bins=30, ax=axes[1])
axes[1].set_title('Sleep Duration Distribution by Late-Night Usage')
axes[1].set_xlabel('Sleep Duration (hours)')
plt.tight_layout()
plt.show()

---
### RQ3 — Do some platforms have higher depression (PHQ-9) scores?
*Method: One-way ANOVA + bar chart*

In [ ]:
platform_stats = df.groupby('primary_platform')['phq9_score'].agg(
    mean='mean', std='std', count='count'
).sort_values('mean', ascending=False).round(2)
print('PHQ-9 by platform:')
print(platform_stats)

groups = [g['phq9_score'].values for _, g in df.groupby('primary_platform')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'\nANOVA — F={f_stat:.3f}, p={p_val:.4f}')
print('Significant (p<0.05):', p_val < 0.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
platform_stats['mean'].plot(kind='bar', color='steelblue', ax=axes[0])
axes[0].set_title('Mean PHQ-9 Score by Platform')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Mean PHQ-9 Score')
axes[0].tick_params(axis='x', rotation=30)

sns.violinplot(data=df, x='primary_platform', y='phq9_score', ax=axes[1])
axes[1].set_title('PHQ-9 Score Distribution by Platform')
axes[1].set_xlabel('Platform')
axes[1].set_ylabel('PHQ-9 Score')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
### RQ4 — Does social comparison trigger affect mental health scores?
*Method: Independent-samples T-Test + Chi-Square (severity vs trigger)*

In [ ]:
trig    = df[df['social_comparison_trigger']]
no_trig = df[~df['social_comparison_trigger']]

for score, label in [('gad7_score', 'GAD-7'), ('phq9_score', 'PHQ-9')]:
    t_stat, p_val = stats.ttest_ind(trig[score], no_trig[score])
    print(f'{label}: trigger mean={trig[score].mean():.2f}, '
          f'no-trigger mean={no_trig[score].mean():.2f}, '
          f't={t_stat:.3f}, p={p_val:.4f}, sig={p_val<0.05}')

# Chi-Square: GAD-7 severity vs social comparison trigger
ct = pd.crosstab(df['gad7_severity'], df['social_comparison_trigger'])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print(f'\nChi-Square (GAD-7 severity × trigger): χ²={chi2:.3f}, p={p_chi:.4f}, dof={dof}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='social_comparison_trigger', y='gad7_score', ax=axes[0])
axes[0].set_title('GAD-7 Score by Social Comparison Trigger')
axes[0].set_xticklabels(['No Trigger', 'Trigger'])

ct_pct = ct.div(ct.sum(axis=1), axis=0)
ct_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('GAD-7 Severity Distribution by Social Comparison')
axes[1].set_xlabel('GAD-7 Severity')
axes[1].set_ylabel('Proportion')
axes[1].legend(['No Trigger', 'Trigger'], title='Social Comparison')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
### RQ5 — Which user archetypes are at highest mental health risk?
*Method: ANOVA + Data Segmentation*

In [ ]:
archetype_stats = df.groupby('user_archetype').agg(
    n=('user_id', 'count'),
    avg_screen_time=('daily_screen_time_hours', 'mean'),
    avg_sleep=('sleep_duration_hours', 'mean'),
    avg_gad7=('gad7_score', 'mean'),
    avg_phq9=('phq9_score', 'mean'),
    pct_late_night=('late_night_usage', 'mean'),
    pct_social_trigger=('social_comparison_trigger', 'mean'),
).round(2).sort_values('avg_gad7', ascending=False)

print('Risk profile by archetype:')
print(archetype_stats.to_string())

groups = [g['gad7_score'].values for _, g in df.groupby('user_archetype')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'\nANOVA (GAD-7 ~ archetype): F={f_stat:.3f}, p={p_val:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
archetype_stats[['avg_gad7', 'avg_phq9']].plot(kind='bar', ax=axes[0], colormap='coolwarm')
axes[0].set_title('Average Mental Health Scores by Archetype')
axes[0].set_xlabel('User Archetype')
axes[0].set_ylabel('Mean Score')
axes[0].tick_params(axis='x', rotation=20)

sns.heatmap(
    archetype_stats[['avg_screen_time', 'avg_sleep', 'avg_gad7', 'avg_phq9',
                     'pct_late_night', 'pct_social_trigger']].T,
    annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1]
)
axes[1].set_title('Risk Factor Heatmap by Archetype')
plt.tight_layout()
plt.show()

---
## Mental Health Dashboard — Summary Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Social Media & Mental Health — Dashboard', fontsize=16, fontweight='bold')

# 1. GAD-7 severity distribution
gad7_counts = df['gad7_severity'].value_counts().reindex(gad7_order)
axes[0, 0].pie(gad7_counts, labels=gad7_order, autopct='%1.1f%%', startangle=90,
               colors=sns.color_palette('YlOrRd', 4))
axes[0, 0].set_title('GAD-7 Anxiety Severity')

# 2. PHQ-9 severity distribution
phq9_counts = df['phq9_severity'].value_counts().reindex(phq9_order)
axes[0, 1].pie(phq9_counts, labels=phq9_order, autopct='%1.1f%%', startangle=90,
               colors=sns.color_palette('Blues', 5))
axes[0, 1].set_title('PHQ-9 Depression Severity')

# 3. Screen time histogram
sns.histplot(df['daily_screen_time_hours'], bins=30, kde=True, ax=axes[0, 2], color='teal')
axes[0, 2].set_title('Daily Screen Time Distribution')
axes[0, 2].set_xlabel('Hours per Day')

# 4. Gender breakdown
gender_counts = df['gender'].value_counts()
axes[1, 0].bar(gender_counts.index, gender_counts.values,
               color=sns.color_palette('pastel')[:len(gender_counts)])
axes[1, 0].set_title('Gender Distribution')
axes[1, 0].set_ylabel('Count')

# 5. Platform usage
platform_counts = df['primary_platform'].value_counts()
axes[1, 1].barh(platform_counts.index, platform_counts.values, color='steelblue')
axes[1, 1].set_title('Primary Platform Usage')
axes[1, 1].set_xlabel('Users')

# 6. Correlation heatmap (numeric columns)
numeric_cols = ['age', 'daily_screen_time_hours', 'sleep_duration_hours', 'gad7_score', 'phq9_score']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 2], vmin=-1, vmax=1)
axes[1, 2].set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig('mental_health_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved to mental_health_dashboard.png')

---
## Key Findings Summary

In [ ]:
print('=== KEY FINDINGS ===')
print(f"Total participants: {len(df):,}")
print(f"Age range: {df['age'].min()}–{df['age'].max()} years")
print(f"Avg daily screen time: {df['daily_screen_time_hours'].mean():.2f} h")
print(f"Avg sleep duration:    {df['sleep_duration_hours'].mean():.2f} h")
print(f"Avg GAD-7 score:       {df['gad7_score'].mean():.2f} / 21")
print(f"Avg PHQ-9 score:       {df['phq9_score'].mean():.2f} / 27")
print()
pct_severe_gad7 = (df['gad7_severity'] >= 'Moderate').mean() * 100
pct_severe_phq9 = (df['phq9_severity'] >= 'Moderate').mean() * 100
print(f"% with Moderate+ anxiety (GAD-7):    {pct_severe_gad7:.1f}%")
print(f"% with Moderate+ depression (PHQ-9): {pct_severe_phq9:.1f}%")
print()
top_risk = df.groupby('user_archetype')['gad7_score'].mean().idxmax()
print(f"Highest-risk archetype (GAD-7): {top_risk}")
top_platform = df.groupby('primary_platform')['phq9_score'].mean().idxmax()
print(f"Highest-risk platform (PHQ-9):  {top_platform}")